In [1]:
import torch
import torch.nn as nn
from torch import tensor

### var

In [41]:
_list = tensor([1., 2., 6.])
_list.var()

tensor(7.)

In [42]:
_list.var(unbiased=False)

tensor(4.6667)

In [43]:
mean = sum(_list) / len(_list)
mean

tensor(3.)

In [44]:
variance = sum((x - mean) ** 2 for x in _list) / (len(_list) - 1)
variance

tensor(7.)

In [45]:
variance = sum((x - mean) ** 2 for x in _list) / len(_list)
variance

tensor(4.6667)

### Use Default

In [68]:
bn = nn.BatchNorm1d(1)

x = torch.tensor([
    [
        [1., 2., 6.]
    ]
])

In [69]:
for p in bn.named_parameters():
    print(p)

('weight', Parameter containing:
tensor([1.], requires_grad=True))
('bias', Parameter containing:
tensor([0.], requires_grad=True))


In [70]:
y_pred = bn(x)
y_pred.mean(), y_pred.var(unbiased=False)

(tensor(0., grad_fn=<MeanBackward0>), tensor(1.0000, grad_fn=<VarBackward0>))

In [71]:
y_pred

tensor([[[-0.9258, -0.4629,  1.3887]]], grad_fn=<NativeBatchNormBackward0>)

In [72]:
y_pred.var(unbiased=False)

tensor(1.0000, grad_fn=<VarBackward0>)

In [73]:
bn.weight, bn.bias

(Parameter containing:
 tensor([1.], requires_grad=True),
 Parameter containing:
 tensor([0.], requires_grad=True))

In [74]:
mean = x.mean(dim=(0, 2), keepdim=True)
mean

tensor([[[3.]]])

In [75]:
var = x.var(dim=(0, 2), unbiased=False, keepdim=True)
var

tensor([[[4.6667]]])

In [116]:
var.item()

4.666666507720947

In [76]:
eps = bn.eps
eps

1e-05

In [77]:
x_norm = (x - mean) / torch.sqrt(var + eps)
x_norm

tensor([[[-0.9258, -0.4629,  1.3887]]])

In [78]:
gamma = bn.weight
beta = bn.bias
gamma, beta

(Parameter containing:
 tensor([1.], requires_grad=True),
 Parameter containing:
 tensor([0.], requires_grad=True))

In [79]:
y = gamma.view(1, -1, 1) * x_norm + beta.view(1, -1, 1)
y

tensor([[[-0.9258, -0.4629,  1.3887]]], grad_fn=<AddBackward0>)

In [80]:
y_pred

tensor([[[-0.9258, -0.4629,  1.3887]]], grad_fn=<NativeBatchNormBackward0>)

### Change Param

In [81]:
bn = nn.BatchNorm1d(1)

with torch.no_grad():
    bn.weight.copy_(torch.tensor([2.]))
    bn.bias.copy_(torch.tensor([3.]))

x = torch.tensor([
    [
        [1., 2., 6.]
    ]
])

y_pred = bn(x)
y_pred.mean(), y_pred.var(unbiased=False)

(tensor(3., grad_fn=<MeanBackward0>), tensor(4.0000, grad_fn=<VarBackward0>))

In [83]:
bn = nn.BatchNorm1d(1)

with torch.no_grad():
    bn.weight.copy_(torch.tensor([1.5]))
    bn.bias.copy_(torch.tensor([3.]))

x = torch.tensor([
    [
        [1., 2., 6.]
    ]
])

y_pred = bn(x)
y_pred.mean(), y_pred.var(unbiased=False)

(tensor(3., grad_fn=<MeanBackward0>), tensor(2.2500, grad_fn=<VarBackward0>))

In [84]:
1.5 ** 2

2.25

### Grad

In [97]:
bn = nn.BatchNorm1d(1)

with torch.no_grad():
    bn.weight.copy_(torch.tensor([2.]))
    bn.bias.copy_(torch.tensor([0.]))

x = torch.tensor([
    [
        [1., 2., 6.]
    ]
])

y_pred = bn(x)
y_pred.mean(), y_pred.var(unbiased=False)

(tensor(0., grad_fn=<MeanBackward0>), tensor(4.0000, grad_fn=<VarBackward0>))

In [98]:
y_pred

tensor([[[-1.8516, -0.9258,  2.7775]]], grad_fn=<NativeBatchNormBackward0>)

In [111]:
[y_pred[0][0][0].item(), y_pred[0][0][1].item(), y_pred[0][0][2].item()]

[-1.8516380786895752, -0.9258190393447876, 2.7774572372436523]

In [99]:
y_pred.shape

torch.Size([1, 1, 3])

In [100]:
y_target = tensor([[[ 2, 0, 2 ]]])

In [101]:
loss_fn = nn.MSELoss()

In [102]:
loss = loss_fn(y_pred, y_target)

In [103]:
loss

tensor(5.4322, grad_fn=<MseLossBackward0>)

In [104]:
loss.backward()

In [105]:
bn.weight.grad, bn.bias.grad

(tensor([3.3828]), tensor([-2.6667]))

In [114]:
bn.weight, bn.bias

(Parameter containing:
 tensor([2.], requires_grad=True),
 Parameter containing:
 tensor([0.], requires_grad=True))

In [124]:
weight = tensor(2.)
bias = tensor(0.)

In [120]:
mean = (x[0] + x[1] + x[2]) / 3
mean

tensor(3.)

In [122]:
variance = ((x[0] - mean) ** 2 + (x[1] - mean) ** 2 + (x[2] - mean) ** 2 ) / (len(x))
variance

tensor(4.6667)

In [123]:
eps = 1e-05
x_norm = (x - mean) / torch.sqrt(variance + eps)
x_norm

tensor([-0.9258, -0.4629,  1.3887])

In [125]:
y_pred = weight * x_norm + bias
y_pred

tensor([-1.8516, -0.9258,  2.7775])

In [130]:
y_target = tensor([ 2, 0, 2 ])

In [132]:
L0 = (y_pred[0] - y_target[0]) ** 2
L1 = (y_pred[1] - y_target[1]) ** 2
L2 = (y_pred[2] - y_target[2]) ** 2
L0, L1, L2

(tensor(14.8351), tensor(0.8571), tensor(0.6044))

In [133]:
L = (L0 + L1 + L2)/3
L

tensor(5.4322)

In [134]:
y_pred_0 = x_norm[0] * weight + bias
y_pred_1 = x_norm[1] * weight + bias
y_pred_2 = x_norm[2] * weight + bias

y_pred_0, y_pred_1, y_pred_2

(tensor(-1.8516), tensor(-0.9258), tensor(2.7775))

In [135]:
dL0_dypred0 = 2 * (y_pred[0] - y_target[0])
dypred0_dweight = x_norm[0]

dL0_dweight = dL0_dypred0 * dypred0_dweight
dL0_dweight

tensor(7.1318)

In [140]:
dL0_dweight = 2 * (y_pred[0] - y_target[0]) * x_norm[0]
dL1_dweight = 2 * (y_pred[1] - y_target[1]) * x_norm[1]
dL2_dweight = 2 * (y_pred[2] - y_target[2]) * x_norm[2]

dL_weight = (dL0_dweight + dL1_dweight + dL2_dweight)/3
dL.item()

3.3827788829803467

In [139]:
dL0_dypred0 = 2 * (y_pred[0] - y_target[0])
dypred0_dbias = 1

dL0_dbias = dL0_dypred0 * dypred0_dbias
dL0_dbias

tensor(-7.7033)

In [143]:
dL0_dbias = 2 * (y_pred[0] - y_target[0])
dL1_dbias = 2 * (y_pred[1] - y_target[1])
dL2_dbias = 2 * (y_pred[2] - y_target[2])

dL_bias = (dL0_dbias + dL1_dbias + dL2_dbias)/3
dL_bias.item()

-2.6666667461395264

In [129]:
((2 * (y_pred[0] - y_target[0]) * x_norm[0] + 2 * (y_pred[1] - y_target[1]) * x_norm[1] + 2 * (y_pred[2] - y_target[2]) * x_norm[2])/3).item()

3.3827788829803467

In [131]:
((2 * (y_pred[0] - y_target[0]) * 1 + 2 * (y_pred[1] - y_target[1]) * 1 + 2 * (y_pred[2] - y_target[2]) * 1)/3).item()

-2.6666667461395264

In [128]:
bn.weight.grad.item(), bn.bias.grad.item()

(3.3827786445617676, -2.6666667461395264)